In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/facebook/bart-large-cnn

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/facebook/bart-large-cnn)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
# Use a pipeline as a high-level helper
# Warning: Pipeline type "summarization" is no longer supported in transformers v5.
# You must load the model directly (see below) or downgrade to v4.x with:
# 'pip install "transformers<5.0.0'
# The model is loaded directly in the next cell (cell `mKSjFEji7ku1`).


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

## Remote Inference via Inference Providers
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you.
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [ ]:
import os
os.environ['HF_TOKEN'] = 'ADD TOKEN HERE'

In [5]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],
)

result = client.summarization(
    "The tower is 324 metres (1,063 ft) tall, about the same height as an 81-storey building, and the tallest structure in Paris. Its base is square, measuring 125 metres (410 ft) on each side. During its construction, the Eiffel Tower surpassed the Washington Monument to become the tallest man-made structure in the world, a title it held for 41 years until the Chrysler Building in New York City was finished in 1930. It was the first structure to reach a height of 300 metres. Due to the addition of a broadcasting aerial at the top of the tower in 1957, it is now taller than the Chrysler Building by 5.2 metres (17 ft). Excluding transmitters, the Eiffel Tower is the second tallest free-standing structure in France after the Millau Viaduct.",
    model="facebook/bart-large-cnn",
)

In [7]:
import pandas as pd
try:
    df = pd.read_csv('/content/preprocessed.csv')
    print("preprocessed.csv loaded successfully.")
except FileNotFoundError:
    print("Error: preprocessed.csv not found. Please ensure the file exists in /content/.")
    exit()

review_column = 'reviewText'

if review_column not in df.columns:
    print(f"Error: Column '{review_column}' not found in preprocessed.csv. Available columns: {df.columns.tolist()}")
    exit()

def count_words(text):
    if isinstance(text, str):
        return len(text.split())
    return 0

df['word_count'] = df[review_column].apply(count_words)
long_reviews_df = df[df['word_count'] > 100]

selected_reviews = long_reviews_df.head(10)

if selected_reviews.empty:
    print("No reviews found with more than 100 words. Please check your data or adjust the word count threshold.")
    exit()

print(f"Selected {len(selected_reviews)} reviews with more than 100 words for summarization.")

summarized_data = []
for index, row in selected_reviews.iterrows():
    review_text = row[review_column]
    if not isinstance(review_text, str):
        print(f"Skipping non-string review at index {index}: {review_text}")
        continue

    inputs = tokenizer([review_text], max_length=1024, truncation=True, return_tensors="pt")

    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=70,
        min_length=40,
        early_stopping=True
    )
    summary = [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=False) for g in summary_ids]

    summarized_data.append({
        'original_review': review_text,
        'original_word_count': row['word_count'],
        'summary': summary[0] if summary else "",
        'summary_word_count': count_words(summary[0]) if summary else 0
    })

summarized_df = pd.DataFrame(summarized_data)

output_filename = 'summarized_reviews.csv'
summarized_df.to_csv(output_filename, index=False)

print(f"Summarized data exported to {output_filename}")
print("First 5 rows of the summarized data:")
print(summarized_df.head())

preprocessed.csv loaded successfully.
Selected 10 reviews with more than 100 words for summarization.
Summarized data exported to summarized_reviews.csv
First 5 rows of the summarized data:
                                     original_review  original_word_count  \
0  I've been using Dreamweaver (and it's predeces...                  214   
1  The demo is done with the PC version, with ref...                  435   
2  I've been creating websites with Dreamweaver f...                  452   
3  I decided (after trying a number of other prod...                  213   
4  I spent several hours on the lesson and I love...                  141   

                                             summary  summary_word_count  
0  For someone who is an experienced web designer...                  56  
1  The demo is done with the PC version, with ref...                  40  
2  This is not an advanced tips and tricks resour...                  56  
3  Learn Adobe Dreamweaver is an excellent comp

In [11]:
import pandas as pd

try:
    summarized_df = pd.read_csv('summarized_reviews.csv')
    print("summarized_reviews.csv loaded successfully.")
except FileNotFoundError:
    print("Error: summarized_reviews.csv not found. Please ensure the file exists.")
    exit()

question_review_text = None
for index, row in summarized_df.iterrows():
    if '?' in str(row['original_review']):
        question_review_text = str(row['original_review'])
        break

if question_review_text:
    selected_review = question_review_text
    print(f"Found a review with a question:\nOriginal Review: {selected_review}")
else:
    if not summarized_df.empty:
        selected_review = summarized_df.iloc[0]['original_review']
        print("No review with an explicit question mark found. Proceeding with the first review for demonstration purposes:")
        print(f"Original Review: {selected_review}")
    else:
        print("The 'summarized_reviews.csv' file is empty. Cannot proceed.")
        exit()

prompt = f"Given the following customer review, please provide a helpful and polite response as a service representative. Firstly ask for sorry and find a quick solution for the customer issue.\n\nCustomer Review: {selected_review}\n\nService Representative Response:"

if 'tokenizer' not in locals() or 'model' not in locals():
    print("Error: 'tokenizer' or 'model' not found. Please run cell mKSjFEji7ku1 first.")
    exit()

inputs = tokenizer([prompt], max_length=1024, truncation=True, return_tensors="pt")

input_len = inputs["input_ids"].shape[1]

response_ids = model.generate(
    inputs["input_ids"],
    num_beams=5,
    max_length=input_len + 200,
    min_length=input_len + 50,
    early_stopping=True
)

generated_response_tokens = response_ids[0][input_len:]
service_response = tokenizer.decode(generated_response_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()

print(f"\nGenerated Service Representative Response:\n{service_response}")

summarized_reviews.csv loaded successfully.
Found a review with a question:
Original Review: I have had Dreamweaver MX2004 since it came out back then. Spent years with it. Feel like I know it real well, but I am still familiar with tables as opposed to CSS. So I thought this would be a great introduction, and it is. The problem is that while I am looking at the video, and that simplifies things a lot, I am not getting the intuitive explanation as to why things work the way they do. I understand just knowing how to work them is sufficient. I'm tempted to delve into the rich full attributes of Dreamweaver CS5 as explained by this video, albeit difficult to understand the more advanced features, but I won't because this is about the video.

The opening salvo is chock full of... for example, this is the URL; this is where you type in the address bar etc. Only when you start to get into areas such as CSS do you get an introduction that for me is beneficial, and that is only because I am so